In [ ]:
skip_training = False # <--- Change me
base_dir = './'
disable_tqdm = False

data_augmentation = True
class_balancing = 'wrs'

batch_size = 64
num_epochs = 20
weight_decay=1e-4
lr=3e-4
layers_to_unfreeze = ''
dropout = None

nb_name = 'vit'

## Setting up our environment

In [8]:
import sys
if 'google.colab' in sys.modules:
    !pip install torch torchvision medmnist matplotlib seaborn scikit-learn tqdm trl datasets transformers -q

In [9]:
# Imports
import os
import time
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, models
from tqdm.notebook import tqdm
import medmnist
from medmnist import INFO
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, RocCurveDisplay
import seaborn as sns
from typing import Dict, List, Tuple
import torch.nn.functional as F
from sklearn.preprocessing import label_binarize


# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"MedMNIST version: {medmnist.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device selected: {device}")

## Part 1: CNNs vs. ViTs (10 points)

In this part, we will fine-tune ResNet-18 and a Vision Transformer (ViT-B-16) on the DermaMNIST dataset and compare their performance.

Your primary tasks in this section are as follows:

- Fine-tune a ResNet-18 model on the DermaMNIST dataset.
- Fine-tune a Vision Transformer (ViT-Base) model on the same dataset.
- Compare the two models in terms of classification performance, computational efficiency, and training dynamics.
- Analyze the results to draw conclusions about the suitability of CNNs and transformer-based models for the given dataset.
  
We provide data loaders, some helper functions and several hints along the way to help you get started. You are expected to make use of these utilities and may modify them if needed.

### Your tasks:

**T1. Model Setup and Training**

Start by loading pre-trained weights (eg. ImageNet) for both models and modify the architectures so they are suitable for DermaMNIST.

Implement a full training pipeline. This should include a data augmentation strategy appropriate for dermoscopic images, a loss function, and an optimizer with clearly specified hyperparameters (learning rate, weight decay, etc.). If you use a learning rate scheduler or regularization techniques such as early stopping or dropout, briefly explain your reasoning and their effect on training.

**T2. Fine-Tuning**

Decide how you will fine-tune each model. You may fine-tune all layers, freeze part of the network and train only the classifier head, or use a progressive unfreezing strategy. Justify your choice based on factors such as dataset size, training stability, overfitting risk, or computational constraints.

During training, track:

- training and validation loss per epoch,
- training and validation accuracy per epoch,
- training time per epoch,
- and the number of trainable versus frozen parameters.

Use these metrics to support your discussion.

**T3. Model Evaluation**

Evaluate both models on the test set using the provided `evaluate()` function. Report with metrics such as classification accuracy, confusion matrix, etc. Use these results to highlight strengths and weaknesses of each model.

**Good to know:** A test accuracy of around 70-80% is achievable for both ViTs and ResNets. Try at least two or three different configurations for both the ResNet and the ViT models. The goal is not to achieve the highest accuracy, but to improve your understanding on which techniques work better/worse for a given situation.

---
**Grading**

You can gain a total of 10 points in this part. 6 points will be assigned if your code is correct and you give a clear and detailed explanation of your design process and design decisions. Try to be detailed but stay to the point, remain professional, and try to remain within 2 pages (less is also ok). Discuss the following:

A. Challenges Encountered

For each model, describe the main challenges you faced (e.g. unstable training, slow convergence, overfitting, etc). Explain what strategies you tried to address them, such as data augmentation, regularization, learning rate changes, or layer freezing. Be clear about what worked and what didn't, and support your claims with evidence from training curves or evaluation accuracy.

B. Model Comparison

Compare ResNet-18 and ViT-Base in terms of classification performance, training dynamics, and computational efficiency. Discuss which model performed better overall and whether certain classes benefited more from one architecture. Some questions that might guide your discussion are as follows. Which model achieved higher accuracy? Are there particular disease classes where one model excels? Did you observe any overfitting while training? How did you address it? Which hyperparameters had the biggest impact on each model? Which model is more practical for deployment in terms of accuracy and training time? What does the confusion matrix tell you about each model's decision-making?

The other 4 points depend on the how good your models are (accuracy), evaluated on the test set. Only correct solutions count for this. If your code contains errors, some points may be deducted for that.

In [ ]:
base_path = os.path.join(base_dir, nb_name)

#### Section 1.1 - Load and understand the data
**DermaMNIST** is a collection of skin lesion images that reflects the real-world challenges of medical AI. Medical datasets tend to be small and expensive to create. Each image needs to be carefully labeled by experts, and certain conditions are naturally rarer than others.

Our dataset contains seven different types of skin lesions, ranging from benign growths to melanoma, the most dangerous form of skin cancer. The training set has around 7,000 images - which is quite typical for medical applications. This constraint is going to be important as we think about which architecture might work better.

Let's look at the seven classes we'll be working with:

| Class | What It Is | Why It Matters |
|-------|------------|----------------|
| Actinic keratoses | Pre-cancerous sun damage | Catching these prevents progression |
| Basal cell carcinoma | Most common skin cancer | Highly curable if detected early |
| Benign keratosis | Harmless growths | Shouldn't be confused with cancer |
| Dermatofibroma | Benign skin nodules | Typically straightforward to diagnose |
| Melanoma | Aggressive skin cancer | Critical to detect - mistakes are costly |
| Melanocytic nevi | Common moles | Benign, but look similar to melanoma |
| Vascular lesions | Blood vessel growths | Usually distinct in appearance |

Here's something to keep in mind as you train your models: the distinction between melanoma and nevi is quite difficult, even for human experts. These two classes are the ones most likely to get confused, and when you look at your confusion matrices later, pay special attention to how your models handle this critical distinction.

**Note:** If you are using a Windows local setup, change `num_workers = 0`. Loading the dataset for the first time might take a bit longer in this case (~10 minutes).

In [14]:
# This is an helper function to create dataloaders
# You can change the augmentation function in this code cell
def get_data_loaders(batch_size: int = 64) -> Tuple[DataLoader, DataLoader, DataLoader, int]:
    """
    Load and prepare the DermaMNIST dataset.

    Returns train, validation, and test loaders plus the number of classes.
    """
    info = INFO['dermamnist']
    n_classes = len(info['label'])
    DataClass = getattr(medmnist, info['python_class'])

    print(f"\n{'='*60}")
    print(f"Loading DermaMNIST Dataset")
    print(f"{'='*60}")
    print(f"\nWe have {n_classes} classes to work with:")
    for idx, name in info['label'].items():
        print(f"  {idx}: {name}")
    print(f"{'='*60}\n")

    g = torch.Generator()
    g.manual_seed(42)

    # normalization
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # mean from ImageNet dataset
        std=[0.229, 0.224, 0.225]    # std from ImageNet dataset
    )

    # Training transform with augmentation
    # feel free to change
    base_transforms = [transforms.ToTensor(), normalize]

    if data_augmentation:
        train_transform = transforms.Compose([
            transforms.RandomRotation(180),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            *base_transforms
        ])
    else:
        train_transform = transforms.Compose(base_transforms)

    # validation/test transform
    eval_transform = transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ])

    # load the datasets
    train_dataset = DataClass(split='train', transform=train_transform, download=True, size=224)
    val_dataset   = DataClass(split='val',   transform=eval_transform, download=True, size=224)
    test_dataset  = DataClass(split='test',  transform=eval_transform, download=True, size=224)

    def collate_fn(batch):
        images, labels = zip(*batch)
        images = torch.stack(images)
        labels = torch.tensor([label.item() if hasattr(label, 'item') else label.flatten()[0]
                              if hasattr(label, 'flatten') else label
                              for label in labels], dtype=torch.long)
        return images, labels
    
    sampler = None
    shuffle = True
    if class_balancing == 'wrs':
        targets = train_dataset.labels.squeeze()
        class_count = np.bincount(targets)
        class_weights = 1.0 / class_count
        sample_weights = torch.from_numpy(class_weights[targets]).double()
        
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True,
            generator=g
        )
        shuffle = False

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=shuffle, num_workers=2, collate_fn=collate_fn, sampler=sampler, generator=g)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=2, collate_fn=collate_fn)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2, collate_fn=collate_fn)

    print(f"Dataset splits:\n")
    print(f"  Training:   {len(train_dataset):,} images")
    print(f"  Validation: {len(val_dataset):,} images")
    print(f"  Test:       {len(test_dataset):,} images\n")

    return train_loader, val_loader, test_loader, n_classes

# the execution might take a while since we load images with size 224 x 224
train_loader, val_loader, test_loader, n_classes = get_data_loaders(batch_size)

In [15]:
class_names = {i: INFO['dermamnist']['label'][str(i)] for i in range(n_classes)}
images, labels = next(iter(train_loader))

print(f"Each batch has shape: {images.shape}")
print(f"That's (batch_size, channels, height, width)\n")

fig, axs = plt.subplots(2, 4, figsize=(14, 7))
axs = axs.ravel()

for i in range(8):
    img = images[i].numpy().transpose((1, 2, 0))
    mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
    img = np.clip(std * img + mean, 0, 1)

    axs[i].imshow(img)
    axs[i].set_title(class_names[labels[i].item()], fontsize=9)
    axs[i].axis('off')

plt.suptitle("Sample Images from DermaMNIST (224×224)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

#### Section 1.2 - Helper functions

##### Evaluate and training history

In [ ]:
# This is an helper function to evaluate your model.
# Do not modify it, unless necessary
def evaluate(model, data_loader, criterion=None, device=device):
    """
    Evaluation function

    Args:
        model: The model being evaluated
        data_loader: DataLoader (validation or test set)
        criterion: Loss function
        device: Device to run on

    Returns:
        all_preds: numpy array of predictions
        all_labels: numpy array of true labels
        all_probs: numpy array of probabilities for all classes
        accuracy: accuracy percentage
        loss: average loss (None if criterion not provided)
    """
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc='Evaluating', leave=False, disable=disable_tqdm):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            if criterion is not None:
                loss = criterion(outputs, labels)
                running_loss += loss.item()
            
            probs = F.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    accuracy = accuracy_score(all_labels, all_preds) * 100.
    avg_loss = running_loss / len(data_loader) if criterion is not None else None

    print(f"######## Test Accuracy: {accuracy:.2f}% ########")

    return all_preds, all_labels, all_probs, accuracy, avg_loss

# This is an helper function to plot the training history.
# Do not modify it, unless necessary
def plot_training_history(history, model_name):
    """Plot training and validation curves"""
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

    # Loss plot
    ax1.plot(history['train_loss'], label='Train Loss', linewidth=2)
    ax1.plot(history['val_loss'], label='Val Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title(f'{model_name} - Loss Curves', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    # Accuracy plot
    ax2.plot(history['train_acc'], label='Train Acc', linewidth=2)
    ax2.plot(history['val_acc'], label='Val Acc', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title(f'{model_name} - Accuracy Curves', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    ax3.plot(history['epoch_times'], label='Time (s)', color='green', linewidth=2)
    ax3.set_xlabel('Epoch', fontsize=12)
    ax3.set_ylabel('Seconds', fontsize=12)
    ax3.set_title(f'{model_name} - Training Time', fontsize=14, fontweight='bold')
    ax3.legend(fontsize=11)
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


##### Train
Write a training loop (or a `train_model` function) that runs for `num_epochs`, computes cross-entropy loss, back-propagates, and records train/val accuracy and loss for each epoch

In [ ]:
def train_model(
    model, train_loader, val_loader, criterion, optimizer, scheduler=None,
    num_epochs=10, device='cpu', checkpoint_path='best_model.pth', unfreeze_epoch=None
):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    
    history = {
        'train_loss': [], 'val_loss': [], 
        'train_acc': [], 'val_acc': [], 
        'epoch_times': [],
        'trainable_params': trainable_params,
        'frozen_params': frozen_params,
    }
    
    print(f"Model initialized with {trainable_params:,} trainable and {frozen_params:,} frozen parameters.")
    
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        if unfreeze_epoch is not None and epoch == unfreeze_epoch:
            for name, param in model.named_parameters():
                if name in layers_to_unfreeze:
                    param.requires_grad = True

        start_time = time.time()
        
        model.train()
        
        running_loss, running_correct, total_samples = 0.0, 0, 0

        for images, labels in tqdm(train_loader, leave=True, desc=f"Epoch {epoch+1}/{num_epochs}", disable=disable_tqdm):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            running_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

        train_loss = running_loss / total_samples
        train_acc = 100.0 * running_correct / total_samples

        _, _, _, val_acc, val_loss = evaluate(model, val_loader, criterion, device)
        
        if scheduler is not None:
            scheduler.step()

        end_time = time.time()
        epoch_duration = end_time - start_time

        # Update history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['epoch_times'].append(epoch_duration)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            print(f"Epoch {epoch}")
            torch.save(model.state_dict(), checkpoint_path)

    return model, history

##### Confusion matrix
Accuracy alone can be misleading - especially on an imbalanced dataset like DermaMNIST. A model that simply learns to predict the majority class most of the time can still achieve a decent accuracy number while not being useful in practice. With a confusion matrix, you can immediately see:

- **Which classes the model struggles with** - are the errors scattered, or does the model systematically confuse certain pairs?
- **The cost of errors** - in a medical setting, not all mistakes are equal. Which misclassifications here would you consider most dangerous?
- **The effect of class imbalance** - do some classes dominate the predictions? What could that tell you about the model's behaviour?

**Hint:** Consider using `confusion_matrix()` from `sklearn.metrics` and visualise with `seaborn.heatmap()`. Label both axes with `class_names`. Check that you're passing `(true_labels, predicted_labels)` in that order.

In [18]:
def plot_confusion_matrix(model_labels, models_preds, model_name):
    plt.figure(figsize=(8, 8))
    # Recall
    cm_true = confusion_matrix(model_labels, models_preds, normalize='true')
    sns.heatmap(
        cm_true,
        annot=True,
        fmt='.2f',
        cmap='Blues',
        xticklabels=[class_names[i] for i in range(n_classes)],
        yticklabels=[class_names[i] for i in range(n_classes)],
    )
    plt.title(f'{model_name}', fontsize=13, fontweight='bold')
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)

    plt.tight_layout()
    plt.show()

#### Section 1.3 -Load and fine-tune a pre-trained ResNet-18 model

**Your task:** Load a ResNet-18 model and fine-tune it with transfer learning for the 7-class skin lesion classification task. Plot the learning curves while training and evaluate the model on the test set.

**Hints:**
- Load the pre-trained ResNet-18 from `torchvision.models` (use `pretrained=True`)
- Remember that the final fully-connected layer of ResNet-18 (`model.fc`) outputs 1000 classes for ImageNet. Do you need to modify this?
- Try different hyperparameter configurations, e.g., optimizer, learning rate, number of epochs, etc., to find the best set
- Save the best model checkpoint (by validation accuracy) using `torch.save(model.state_dict(), ...)`
- For plotting the learning curves you can check the `plot_training_history` helper function
- For evaluation on the test set you can use the `evaluate()` helper function, passing in `test_loader`


In [ ]:
model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
for param in model.parameters():
    param.requires_grad = False

in_features = model.heads.head.in_features
model.heads.head = nn.Sequential(
    nn.Dropout(p=dropout) if dropout else nn.Identity(),
    nn.Linear(in_features, n_classes)
)
model = model.to(device)

class_weights_tensor = None
if class_balancing == 'wcel':
    train_targets = np.array(train_loader.dataset.labels).reshape(-1)
    class_counts = np.bincount(train_targets, minlength=n_classes)
    class_weights = 1.0 / class_counts
    class_weights = class_weights / class_weights.sum() * n_classes
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)
    print('Class counts:', class_counts)
    print('Class weights:', class_weights.round(4))
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
else:
    criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW([
    {'params': model.heads.head.parameters(), 'lr': lr},
], weight_decay=weight_decay)

history = None
if skip_training:
    with open(f"{base_path}.json", 'r') as f:
        history = json.load(f)

    model.load_state_dict(torch.load(f"{base_path}.pth", map_location=device))
    print('Loaded checkpoint (skip_training=True)')
else:
    model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        num_epochs=num_epochs,
        device=device,
        checkpoint_path=f"{base_path}.pth",
    )
    with open(f"{base_path}.json", 'w') as f:
        json.dump(history, f)

In [ ]:
plot_training_history(history, nb_name)

In [ ]:
preds, labels, probs, acc, test_loss = evaluate(
    model,
    test_loader,
    criterion=criterion,
    device=device
 )
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {acc:.2f}%')

In [ ]:
plot_confusion_matrix(labels, preds, nb_name)

In [ ]:
print(classification_report(
    labels, 
    preds, 
    target_names=[class_names[i] for i in range(n_classes)]
))

In [ ]:
class_idx = int(next(k for k, v in INFO['dermamnist']['label'].items() if v == 'melanoma'))

RocCurveDisplay.from_predictions(
    y_true=(labels == class_idx).astype(int),
    y_pred=probs[:, class_idx],
    name="Melanoma",
)

plt.title("ROC Curve - Melanoma")
plt.grid(alpha=0.3)
plt.show()